# Dispersion analysis from `pinn_data.mat`
This notebook now produces **multiple plots**: time-space field, time traces, 2D f-k map, and extracted dispersion ridge.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat

In [ ]:
mat = loadmat('../Result/pinn_data.mat')
print('Available keys:', [k for k in mat.keys() if not k.startswith('__')])

In [ ]:
# Robust displacement extraction
preferred = ['x', 'disp', 'u', 'response', 'X', 'U']
X = None
selected_key = None

for key in preferred:
    if key in mat:
        arr = np.array(mat[key])
        if np.issubdtype(arr.dtype, np.number) and arr.ndim >= 2:
            X, selected_key = arr, key
            break

if X is None:
    candidates = []
    for k, v in mat.items():
        if k.startswith('__'):
            continue
        arr = np.array(v)
        if np.issubdtype(arr.dtype, np.number) and arr.ndim >= 2:
            candidates.append((k, arr))
    if not candidates:
        raise ValueError('No numeric 2D+ array found in pinn_data.mat. Please inspect manually.')
    selected_key, X = max(candidates, key=lambda kv: kv[1].shape[0] * kv[1].shape[1])

print('Using key:', selected_key, 'raw shape:', X.shape)

if X.ndim > 2:
    X = np.reshape(X, (X.shape[0], -1))
if X.shape[0] < X.shape[1]:
    X = X.T

X = X - X.mean(axis=0, keepdims=True)
nt, nx = X.shape
print('Working shape [time, space]:', X.shape)

In [ ]:
dt = float(np.squeeze(mat['dt'])) if 'dt' in mat else 1.0
dx = float(np.squeeze(mat['dx'])) if 'dx' in mat else 1.0
time = np.arange(nt) * dt
space = np.arange(nx) * dx

## Plot 1: Time-space response field

In [ ]:
plt.figure(figsize=(8,4.5))
plt.pcolormesh(space, time, X, shading='auto', cmap='RdBu_r')
plt.xlabel('Space index / coordinate')
plt.ylabel('Time')
plt.title('PINN response field')
plt.colorbar(label='Displacement')
plt.tight_layout()
plt.show()

## Plot 2: Representative time traces

In [ ]:
idxs = np.linspace(0, nx-1, min(4, nx), dtype=int)
plt.figure(figsize=(8,4))
for j in idxs:
    plt.plot(time, X[:,j], label=f'space #{j}')
plt.xlabel('Time')
plt.ylabel('Displacement')
plt.title('Time traces at selected spatial points')
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Plot 3: f-k dispersion heatmap

In [ ]:
S = np.fft.fftshift(np.fft.fft2(X))
P = np.abs(S)**2
f = np.fft.fftshift(np.fft.fftfreq(nt, d=dt))
k = 2*np.pi*np.fft.fftshift(np.fft.fftfreq(nx, d=dx))
P_db = 10*np.log10(P/(P.max()+1e-12) + 1e-12)

plt.figure(figsize=(8,5))
plt.pcolormesh(k, f, P_db, shading='auto', cmap='magma')
plt.xlabel('Wavenumber k [rad/unit]')
plt.ylabel('Frequency f [1/unit]')
plt.title('Dispersion-style f-k map')
plt.colorbar(label='Power (dB, normalized)')
plt.tight_layout()
plt.show()

## Plot 4: Extracted dispersion ridge (dominant k for each f)

In [ ]:
# Use nonnegative frequencies only
mask_f = f >= 0
f_pos = f[mask_f]
P_pos = P[mask_f, :]

k_ridge = k[np.argmax(P_pos, axis=1)]

plt.figure(figsize=(7,4.5))
plt.plot(k_ridge, f_pos, '.', ms=3)
plt.xlabel('Dominant wavenumber k')
plt.ylabel('Frequency f')
plt.title('Extracted dispersion ridge (argmax over k)')
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### Should you change excitation frequency and replot?
**Yes.** Changing input excitation frequency is recommended, because it excites different frequency bands and helps reveal more complete dispersion branches.

## Plot 5: Combined 2x2 panel (same four figures, no external script call)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# (1) Time-space field
ax = axes[0, 0]
im1 = ax.pcolormesh(space, time, X, shading='auto', cmap='RdBu_r')
ax.set_title('01_time_space_field')
ax.set_xlabel('Space index / coordinate')
ax.set_ylabel('Time')
fig.colorbar(im1, ax=ax, label='Displacement')

# (2) Time traces
ax = axes[0, 1]
for j in np.linspace(0, nx-1, min(4, nx), dtype=int):
    ax.plot(time, X[:, j], label=f'space #{j}')
ax.set_title('02_time_traces')
ax.set_xlabel('Time')
ax.set_ylabel('Displacement')
ax.grid(alpha=0.3)
ax.legend(fontsize=8)

# (3) f-k map
ax = axes[1, 0]
im2 = ax.pcolormesh(k, f, P_db, shading='auto', cmap='magma')
ax.set_title('03_fk_map')
ax.set_xlabel('Wavenumber k [rad/unit]')
ax.set_ylabel('Frequency f [1/unit]')
fig.colorbar(im2, ax=ax, label='Power (dB, normalized)')

# (4) Dispersion ridge
ax = axes[1, 1]
ax.plot(k_ridge, f_pos, '.', ms=3)
ax.set_title('04_dispersion_ridge')
ax.set_xlabel('Dominant wavenumber k')
ax.set_ylabel('Frequency f')
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()